# This notebook describes the workflow for creating summary stats from the data necessary for ML modeling


## Set up your environment
You will want to set your Cloud compute profile to have: 32 CPUs, 208GB RAM, 1000GB Disk (note that this is at the bottom) and the Compute type can be a standard vm



In [ ]:
## Install and import all required libraries
import glob
import numpy as np
import h5py
import pandas as pd
import json
import time
from google.cloud import storage
import glob
import os
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime, timedelta
from scipy import stats
import subprocess
!pip install ujson
import ujson

## Calculate Summaries

In [ ]:
# Define helper functions for BMI extraction: outlier removal (IQR method) and weight-height to BMI conversion
def _remove_outliers_iqr_per_person(df, value_col, id_col="person_id", k=1.5, min_n=4):
    df = df.copy()

    def _filter(g):
        # Don't filter if too few points
        if g[value_col].notna().sum() < min_n:
            return g

        q1 = g[value_col].quantile(0.25)
        q3 = g[value_col].quantile(0.75)
        iqr = q3 - q1

        # If IQR is 0 (all identical), keep everything
        if iqr == 0 or pd.isna(iqr):
            return g

        low = q1 - k * iqr
        high = q3 + k * iqr
        return g[(g[value_col] >= low) & (g[value_col] <= high)]

    return df.groupby(id_col, group_keys=False).apply(_filter)

def _merge_bmi_weight_height(wh_df, bmi_df, min_bmi_val=10, max_bmi_val=80):
    wh_df = wh_df.loc[wh_df['unit_source_value'] != 'No matching concept'].copy()
    wh_df['value_in_kg_cm'] = np.nan
    wh_df.loc[wh_df['unit_source_value'] == 'g', 'value_in_kg_cm'] = wh_df['value_as_number'].loc[
                                                                         wh_df['unit_source_value'] == 'g'] / 1000
    wh_df.loc[wh_df['unit_source_value'] == '[lb_us]', 'value_in_kg_cm'] = wh_df['value_as_number'].loc[wh_df[
                                                                                                            'unit_source_value'] == '[lb_us]'] / 2.205
    wh_df.loc[wh_df['unit_source_value'] == '[oz_av]', 'value_in_kg_cm'] = wh_df['value_as_number'].loc[wh_df[
                                                                                                            'unit_source_value'] == '[oz_av]'] / 35.274
    wh_df.loc[wh_df['unit_source_value'] == '[in_i]', 'value_in_kg_cm'] = wh_df['value_as_number'].loc[wh_df[
                                                                                                           'unit_source_value'] == '[in_i]'] * 2.54
    wh_df.loc[wh_df['unit_source_value'] == '[ft_i]', 'value_in_kg_cm'] = wh_df['value_as_number'].loc[wh_df[
                                                                                                           'unit_source_value'] == '[ft_i]'] * 30.48

    wh_df['date'] = pd.to_datetime(wh_df['measurement_date']).dt.tz_localize(None)
    wh_df["year"] = wh_df['date'].dt.year

    wt_df = wh_df.loc[wh_df['standard_concept_name'] == 'Body weight'].copy()
    ht_df = wh_df.loc[wh_df['standard_concept_name'] == 'Body height'].copy()

    wt_df = wt_df.rename(columns={"date": "wt_date"}).sort_values(["wt_date", "person_id", "year"],
                                                                  kind="mergesort").reset_index(drop=True)
    ht_df = ht_df.rename(columns={"date": "ht_date"}).sort_values(["ht_date", "person_id", "year"],
                                                                  kind="mergesort").reset_index(drop=True)

    print('Transforming weight-height pairs to BMI')
    wh_df = pd.merge_asof(
        wt_df,
        ht_df[["person_id", "year", "ht_date", "value_in_kg_cm", "unit_source_value", "value_as_number"]],
        left_on="wt_date",
        right_on="ht_date",
        by=["person_id", "year"],
        direction="nearest",
    )

    wh_df['bmi'] = wh_df['value_in_kg_cm_x'] / ((wh_df['value_in_kg_cm_y'] / 100) ** 2)
    wh_df = wh_df.loc[np.logical_and(~wh_df['bmi'].isnull(), wh_df['bmi'] < np.inf)]
    wh_df['date'] = wh_df[['wt_date', 'ht_date']].max(axis=1)

    bmi_df['date'] = pd.to_datetime(bmi_df['measurement_date']).dt.tz_localize(None)

    bmi_all_df = pd.concat([bmi_df[['person_id', 'measurement_date', 'value_as_number']].rename(
        columns={'measurement_date': 'date', 'value_as_number': 'bmi'}),
        wh_df[['person_id', 'date', 'bmi']]])

    print('Filtering BMI outliers per-person')
    bmi_all_df = _remove_outliers_iqr_per_person(bmi_all_df, value_col="bmi")

    print(
        f'Filtering out unreasonable BMI values (BMI<{min_bmi_val} or BMI>{max_bmi_val}, defined as optional variables in the function call: min_bmi_val, max_bmi_val)')
    bmi_all_df = bmi_all_df.loc[np.logical_and(bmi_all_df['bmi'] >= min_bmi_val, bmi_all_df['bmi'] <= max_bmi_val)]

    return bmi_all_df

In [ ]:
# Extract weight and height measurements from EHR 
import pandas
import os

# This query represents dataset "Fitbit EHR weight height" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_27640508_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_date,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (3025315, 3036277)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_fitbit = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_fitbit_heart_rate_level = 1 ) )
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_27640508_measurement_df = pandas.read_gbq(
    dataset_27640508_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_27640508_measurement_df.head(5)

In [ ]:
# Extract direct BMI measurements from EHR 
import pandas
import os

# This query represents dataset "Fitbit BMI" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_77080212_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_date,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (3038553)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_fitbit = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_fitbit_heart_rate_level = 1 ) )
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
            ON measurement.measurement_type_concept_id = m_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
            ON measurement.operator_concept_id = m_operator.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
            ON measurement.value_as_concept_id = m_value.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIn
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON measurement.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
            ON v.visit_concept_id = m_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_77080212_measurement_df = pandas.read_gbq(
    dataset_77080212_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_77080212_measurement_df.head(5)

In [ ]:
# Process BMI: convert units, merge weight-height pairs, calculate BMI, remove outliers, filter range 10-80
wh_df = dataset_27640508_measurement_df
bmi_df = dataset_77080212_measurement_df

fitbit_bmi_filt_clean = _merge_bmi_weight_height(wh_df, bmi_df, min_bmi_val=10, max_bmi_val=80)
fitbit_bmi_filt_clean = fitbit_bmi_filt_clean.rename(columns={"date": "bmi_date"})
print(f"Clean BMI records: {len(fitbit_bmi_filt_clean)}")
fitbit_bmi_filt_clean['bmi'].hist(bins=50)

In [ ]:
# Main feature extraction function — iterates through each day in HDF5 file and computes HR, steps, sleep, and activity features
def process_rolling_new(qc_file, dob, vl):

    person_id = os.path.basename(qc_file).replace(".hd5", "")

    if os.path.exists(qc_file):

        with h5py.File(qc_file, 'r') as a:

            for jjk in range(len(sorted(a.keys()))):

                missing_sleep_flag = False
                len_mismatch = False
                date_str = sorted(a.keys())[jjk]
                ref_date = datetime.strptime(date_str[0:10], "%Y-%m-%d")
                age = ref_date.year - dob.year - ((ref_date.month, ref_date.day) < (dob.month, dob.day))
                max_hr_ref = 220 - age

                date_obj = pd.to_datetime(date_str)
                day_of_week = date_obj.day_name()
                day_of_month = date_obj.day
                month = date_obj.month
                year = date_obj.year

                hr_array = a[date_str]['hr_array_qc'][()]
                steps_array = a[date_str]['steps_array_qc'][()]
                sleep_array = a[date_str]['sleep_array'][()]

                if hr_array.size == 0 or steps_array.size == 0:
                    continue

                if hr_array.size > 0 and steps_array.size > 0:
                    step_nw = steps_array.copy().astype(float)
                    step_nw[np.isnan(hr_array)] = np.nan

                    if sleep_array.size > 0:
                        sleep_array[~np.isnan(sleep_array)] = 1
                        sleep_array[np.isnan(sleep_array)] = 0
                    else:
                        sleep_array = np.full(len(hr_array), 0)

                    if np.sum(sleep_array) == 0:
                        missing_sleep_flag = True

                    if len(sleep_array) == len(hr_array):
                        heart_rate_non_sleep = np.where(sleep_array == 0, hr_array, np.nan)
                        heart_rate_sleep     = np.where(sleep_array == 1, hr_array, np.nan)
                        steps_non_sleep      = np.where(sleep_array == 0, steps_array, np.nan)
                        steps_sleep          = np.where(sleep_array == 1, steps_array, np.nan)
                        steps_nw_non_sleep   = np.where(sleep_array == 0, step_nw, np.nan)
                        steps_nw_sleep       = np.where(sleep_array == 1, step_nw, np.nan)
                    else:
                        len_mismatch = True
                        continue

                    # Summary HR/steps info
                    RHR = stats.mode(hr_array, nan_policy='omit').mode
                    RHR_sleep = stats.mode(heart_rate_sleep, nan_policy='omit').mode
                    RHR_non_sleep = stats.mode(heart_rate_non_sleep, nan_policy='omit').mode

                    mean_sleep_hr = np.nanmean(heart_rate_sleep)
                    mean_non_sleep_hr = np.nanmean(heart_rate_non_sleep)
                    mean_hr = np.nanmean(hr_array)

                    # Std deviations for sleep and wake HR
                    std_sleep_hr = np.nanstd(heart_rate_sleep)
                    std_non_sleep_hr = np.nanstd(heart_rate_non_sleep)

                    step_count = np.nansum(steps_array)
                    step_nw_count = np.nansum(step_nw)
                    sleep_duration = np.nansum(sleep_array)

                    # Awake and asleep durations
                    if not missing_sleep_flag:
                        awake_minutes = float(np.nansum(sleep_array == 0))
                        asleep_grpd_rstls_minutes = float(np.nansum(sleep_array == 1))
                    else:
                        awake_minutes = float('nan')
                        asleep_grpd_rstls_minutes = float('nan')

                    # Sleep start time (last awake-to-asleep transition = bedtime)
                    if not missing_sleep_flag:
                        transitions = np.diff(sleep_array)
                        sleep_onsets = np.where(transitions == 1)[0] + 1
                        if len(sleep_onsets) > 0:
                            sleep_start_time = int(sleep_onsets[-1])
                        else:
                            sleep_start_time = float('nan')
                    else:
                        sleep_start_time = float('nan')

                    N_hr_non_sleep = np.sum(~np.isnan(heart_rate_non_sleep))
                    N_hr_sleep     = np.sum(~np.isnan(heart_rate_sleep))
                    N_hr           = np.sum(~np.isnan(hr_array))

                    N_steps = np.sum(~np.isnan(steps_array))
                    N_steps_nw = np.sum(~np.isnan(step_nw))

                    # Pre-classification arrays
                    hr_clean = heart_rate_non_sleep.astype(float)
                    st_clean = steps_nw_non_sleep.astype(float)

                    nan_time = np.sum(np.where(np.isnan(hr_clean), 1, 0).astype(float))

                    max_mvg_ref_mvg = (40 * (max_hr_ref - RHR)) / 100
                    max_mvg_ref_sd  = (20 * (max_hr_ref - RHR)) / 100

                    # % awake time in low/medium/high HR buckets
                    valid_awake_hr = ~np.isnan(hr_clean)
                    n_valid_awake_hr = np.sum(valid_awake_hr)

                    if n_valid_awake_hr > 0:
                        low_hr_threshold = max_mvg_ref_sd + RHR
                        high_hr_threshold = max_mvg_ref_mvg + RHR

                        awake_hr_low_count  = np.sum(valid_awake_hr & (hr_clean < low_hr_threshold))
                        awake_hr_med_count  = np.sum(valid_awake_hr & (hr_clean >= low_hr_threshold) & (hr_clean < high_hr_threshold))
                        awake_hr_high_count = np.sum(valid_awake_hr & (hr_clean >= high_hr_threshold))

                        pct_awake_hr_low  = float(awake_hr_low_count / n_valid_awake_hr)
                        pct_awake_hr_med  = float(awake_hr_med_count / n_valid_awake_hr)
                        pct_awake_hr_high = float(awake_hr_high_count / n_valid_awake_hr)
                    else:
                        pct_awake_hr_low  = float('nan')
                        pct_awake_hr_med  = float('nan')
                        pct_awake_hr_high = float('nan')

                    # 1-min activity type markers
                    mvpa_1min       = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))
                    mvpa_20_1min    = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 20)).astype(float))
                    mvpa_100_1min   = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean > 100)).astype(float))

                    lact_1min       = np.where(np.isnan(hr_clean), np.nan,
                        ((hr_clean > (max_mvg_ref_sd+RHR)) & (hr_clean < (max_mvg_ref_mvg+RHR)) & (st_clean > 0)).astype(float))

                    sedentary_1min  = np.where(np.isnan(hr_clean), np.nan,
                                        ((hr_clean <= (max_mvg_ref_sd+RHR)) & (st_clean == 0)).astype(float))

                    HRR_40_s0_1min     = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean >= (max_mvg_ref_mvg+RHR)) & (st_clean == 0)).astype(float))

                    HRR_20_40_s0_1min  = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean > (max_mvg_ref_sd+RHR)) & (hr_clean < (max_mvg_ref_mvg+RHR)) & (st_clean == 0)).astype(float))

                    HRR_20_sn0_1min    = np.where(np.isnan(hr_clean), np.nan,
                                            ((hr_clean <= (max_mvg_ref_sd+RHR)) & (st_clean > 0)).astype(float))

                    # Build metrics dictionary for this day
                    metrics_this_day = {

                        "mvpa": {
                            "standard": {
                                "raw": int(np.nansum(mvpa_1min == 1)),
                            },
                            "20_plus": {
                                "raw": int(np.nansum(mvpa_20_1min == 1)),
                            },
                            "100_plus": {
                                "raw": int(np.nansum(mvpa_100_1min == 1)),
                            },
                        },
                        "lact": {
                            "raw": int(np.nansum(lact_1min == 1)),
                        },
                        "sedentary": {
                            "raw": int(np.nansum(sedentary_1min == 1)),
                        },
                        "HRR": {
                            "40_s0": {
                                "raw": int(np.nansum(HRR_40_s0_1min == 1)),
                            },
                            "20_40_s0": {
                                "raw": int(np.nansum(HRR_20_40_s0_1min == 1)),
                            },
                            "20_sn0": {
                                "raw": int(np.nansum(HRR_20_sn0_1min == 1)),
                            },
                        },
                        "summary": {
                            "RHR": float(RHR),
                            "RHR_sleep": float(RHR_sleep),
                            "RHR_non_sleep": float(RHR_non_sleep),
                            "mean_sleep_hr": float(mean_sleep_hr),
                            "mean_non_sleep_hr": float(mean_non_sleep_hr),
                            "mean_hr": float(mean_hr),
                            "std_sleep_hr": float(std_sleep_hr),
                            "std_non_sleep_hr": float(std_non_sleep_hr),
                            "step_count": float(step_count),
                            "step_count_nw": float(step_nw_count),
                            "sleep_duration": float(sleep_duration),
                            "awake_minutes": float(awake_minutes),
                            "asleep_grpd_rstls": float(asleep_grpd_rstls_minutes),
                            "sleep_start_time": float(sleep_start_time) if not isinstance(sleep_start_time, float) else sleep_start_time,
                            "pct_awake_hr_low": float(pct_awake_hr_low),
                            "pct_awake_hr_med": float(pct_awake_hr_med),
                            "pct_awake_hr_high": float(pct_awake_hr_high),
                            "N_hr": int(N_hr),
                            "N_hr_sleep": int(N_hr_sleep),
                            "N_hr_non_sleep": int(N_hr_non_sleep),
                            "N_steps": int(N_steps),
                            "N_steps_nw": int(N_steps_nw),
                            "non-wear": float(nan_time),
                            "age": int(age),
                        },
                        "flags": {
                            "missing_sleep": missing_sleep_flag,
                            "len_mismatch": len_mismatch,
                        },
                        "date_info": {
                            "day_of_month": day_of_month,
                            "month": month,
                            "year": year,
                        }
                    }

                    vl['person_id'].append(person_id)
                    vl['date'].append(date_str[:10])
                    vl['metrics'].append(metrics_this_day)

    return vl

In [ ]:
# Wrapper function for parallel execution — extracts DOB and calls feature extraction per person
def process_rolling_new_calculation(qc_file,df_summary_i):
    
    """
    Processes heart rate and step data from HDF5 files to compute activity classifications
    and organize data across multiple time scales.
    
    Parameters:
    -----------
    qc_file : str
        Path to HDF5 file containing quality-controlled heart rate and step data
        Structure: {date_string: {'hr_array_qc_int': array, 'steps_array_qc_int': array}}
        
    df_summary_i : pandas.DataFrame
        DataFrame containing dob detail
        
    """
    
    person_id = os.path.basename(qc_file).replace(".hd5", "")
    dob_str = df_summary_i[df_summary_i['person_id']==int(person_id)]['date_of_birth'].values[0][0:10]
    dob = datetime.strptime(dob_str, "%Y-%m-%d")
    
    activity_types = ['date','person_id','metrics']
    all_act_lvl = {key: [] for key in activity_types}
    

    
    all_act_lvl = process_rolling_new(qc_file, dob, all_act_lvl)
    
    
    
    
    
    key_list = ['date','person_id','metrics']
    act_lvl_dict = {key: all_act_lvl[key] for key in key_list}
    

    return act_lvl_dict, f"jsons_rolling/{person_id}.json"

Here we use multiprocessing library to speed up the calculations and loop through all chunks one by one and save jsons


In [ ]:
# Set GCS bucket paths for reading ingested data and saving outputs
bucket_ = os.environ.get('WORKSPACE_BUCKET')


# ## Ideally we want this setup:
# #paths that are hardcoded to save files generated
# file_paths_values = pd.read_csv('filepaths_values.csv')
# file_paths_values = file_paths_values[file_paths_values['notebook']=='trends_calculation'][['notebook','key_','value_']]
# paths_dict = file_paths_values.set_index("key_")["value_"]

# ## for now we will use the exact read paths, and maybe new paths where you save things
paths_dict = {}
paths_dict['folder_ingest_ehr'] = 'ingested_data_with_ehr_second_level_non_interpolated'
paths_dict['folder_ingest_wehr'] = 'ingested_data_without_ehr_second_level_non_interpolated'
paths_dict['df_save_ehr'] = 'deepak_experiments_ml_df_ehr'#change as you want
paths_dict['df_save_wehr'] = 'deepak_experiments_ml_df_without_ehr'#change as you want

In [ ]:
# Copy HDF5 files from bucket, process all participants in parallel, write per-person JSONs

# DataFrame containing dob details
df_summary_i = pd.read_csv(f'{bucket_}/stratification_person_stats.tsv', sep='\t')

folder_to_copy_from = paths_dict.get("folder_ingest_ehr")
file_path_to_save = paths_dict.get("df_save_ehr")

qc_folder = "all_qc_local"
os.makedirs(qc_folder, exist_ok=True)
os.makedirs("jsons_rolling", exist_ok=True)

# Copy all .hd5 files (~230 GB, will take ~15-30 min)
# subprocess.run(f"gsutil -m cp -r {bucket_}/{folder_to_copy_from}/all_qc/ {qc_folder}/", shell=True, check=True)

# Check subfolder structure
print("Files found:")
subprocess.run(f"ls -d {qc_folder}/*/ | head -10", shell=True)

# Recursive glob since files are in subdirectories
qc_files = sorted(glob.glob(f"{qc_folder}/**/*.hd5", recursive=True))
print(f"Total .hd5 files found: {len(qc_files)}")

# qc_files = qc_files[:100]  # For Validation

st = time.time()
num_cores = multiprocessing.cpu_count()

json_data_v = Parallel(n_jobs=num_cores)(delayed(process_rolling_new_calculation)(qc_file, df_summary_i) for qc_file in tqdm(qc_files))
ed = time.time()
print(f"Processing time: {ed - st:.1f}s")

st = time.time()
print('start_writing')

def make_json_serializable(o):
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, dict):
        return {k: make_json_serializable(v) for k, v in o.items()}
    if isinstance(o, list):
        return [make_json_serializable(x) for x in o]
    return o

def write_json_file(json_data, filename):
    json_data = make_json_serializable(json_data)
    with open(filename, "w") as json_file:
        json_file.write(ujson.dumps(json_data))

Parallel(n_jobs=num_cores)(delayed(write_json_file)(json_v, name)
                          for json_v, name in zip([sublist[0] for sublist in json_data_v],
                                                  [sublist[1] for sublist in json_data_v]))
ed = time.time()
print(f"Writing time: {ed - st:.1f}s")

In [ ]:
# Read per-person JSONs, flatten nested metrics, build single DataFrame, save as parquet
def flatten_dict(d, parent_key="", sep="."):
    """Flatten nested dictionaries for building a DataFrame."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

all_json_files = os.listdir('jsons_rolling')
ehr_p_ids = sorted([int(i.split('.')[0]) for i in all_json_files if "json" in i])

all_rows = []

for p_id in tqdm(ehr_p_ids):
    
    filename = os.path.join('jsons_rolling', f'{p_id}.json')
    if not os.path.exists(filename):
        continue

    with open(filename, "r") as json_file:
        cur_dict = json.load(json_file)

    person_ids = cur_dict["person_id"]
    dates      = cur_dict["date"]
    metrics    = cur_dict["metrics"]

    for pid, date, met in zip(person_ids, dates, metrics):

        flat = flatten_dict(met)     # flatten hierarchical metrics
        flat["person_id"] = int(pid)
        flat["date"] = date

        all_rows.append(flat)

# Make DataFrame
daily_activity_stats_ehr = pd.DataFrame(all_rows)
daily_activity_stats_ehr.to_parquet(f'{bucket_}/{file_path_to_save}.pq')

In [ ]:
# Merge BMI (most recent within 5 years via merge_asof) and is_male into the DataFrame

# Make sure date columns are datetime
daily_activity_stats_ehr['date'] = pd.to_datetime(daily_activity_stats_ehr['date'])
fitbit_bmi_filt_clean['bmi_date'] = pd.to_datetime(fitbit_bmi_filt_clean['bmi_date'])

# Merge BMI — matches each Fitbit day to the most recent BMI within 5 years
daily_activity_stats_ehr = pd.merge_asof(
    daily_activity_stats_ehr.astype({'person_id': int}).sort_values(["date", "person_id"], kind="mergesort").reset_index(drop=True),
    fitbit_bmi_filt_clean[["person_id", "bmi_date", "bmi"]].astype({'person_id': int}).sort_values(["bmi_date", "person_id"], kind="mergesort").reset_index(drop=True),
    left_on="date",
    right_on="bmi_date",
    by="person_id",
    direction="backward",
    tolerance=pd.Timedelta("1825D")  # allow up to 5 years before fitbit measurement
)

# Also add is_male from stratification file
df_summary_i['is_male'] = (df_summary_i['sex_at_birth'] == 'Male').astype(int)
daily_activity_stats_ehr = daily_activity_stats_ehr.merge(
    df_summary_i[['person_id', 'is_male']],
    on='person_id',
    how='left'
)

# Check results
print(f"BMI NaN rate: {daily_activity_stats_ehr['bmi'].isna().mean()*100:.1f}%")
print(f"is_male NaN rate: {daily_activity_stats_ehr['is_male'].isna().mean()*100:.1f}%")
print(f"\nBMI stats:")
print(daily_activity_stats_ehr['bmi'].describe())
print(f"\nis_male distribution:")
print(daily_activity_stats_ehr['is_male'].value_counts())

In [ ]:
# Validate all 14 expected features exist and print summary statistics
latent_cols = {
    'summary.step_count_nw': 'latent_0 (steps_count)',
    'summary.mean_sleep_hr': 'latent_1 (sleep_hr_mean_all)',
    'summary.std_sleep_hr': 'latent_2 (sleep_hr_std)',
    'summary.mean_non_sleep_hr': 'latent_3 (wake_hr_mean_all)',
    'summary.std_non_sleep_hr': 'latent_4 (wake_hr_std)',
    'summary.awake_minutes': 'latent_5 (awake)',
    'summary.asleep_grpd_rstls': 'latent_6 (asleep_grpd_rstls)',
    'summary.age': 'latent_7 (age)',
    'bmi': 'latent_8 (bmi)',
    'is_male': 'latent_9 (is_male)',
    'summary.sleep_start_time': 'sleep_start_time',
    'summary.pct_awake_hr_low': 'pct_awake_hr_low',
    'summary.pct_awake_hr_med': 'pct_awake_hr_med',
    'summary.pct_awake_hr_high': 'pct_awake_hr_high',
}
# Check which columns exist
print("=== Column Check ===")
for col, label in latent_cols.items():
    status = "FOUND" if col in daily_activity_stats_ehr.columns else "MISSING"
    print(f"  {status}: {col}  →  {label}")
# Show stats for all found columns
found = [c for c in latent_cols.keys() if c in daily_activity_stats_ehr.columns]
print(f"\n=== Stats for {len(found)}/{len(latent_cols)} columns ===")
daily_activity_stats_ehr[found].describe()

In [ ]:
# Plot distributions for all features and run sanity checks (awake+asleep=1440, HR buckets sum to 1.0)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (16, 16)
fig, axes = plt.subplots(4, 4, figsize=(20, 18))
fig.suptitle(f'Latent Feature Validation — {daily_activity_stats_ehr["person_id"].nunique()} People', fontsize=16, fontweight='bold')
plots = [
    ('summary.step_count_nw', 'Steps (non-wear adjusted)', 'Expected: 5K-10K avg'),
    ('summary.mean_sleep_hr', 'Mean Sleep HR (bpm)', 'Expected: 55-70 bpm'),
    ('summary.std_sleep_hr', 'Std Sleep HR (bpm)', 'Expected: 3-6 bpm'),
    ('summary.mean_non_sleep_hr', 'Mean Wake HR (bpm)', 'Expected: 70-85 bpm'),
    ('summary.std_non_sleep_hr', 'Std Wake HR (bpm)', 'Expected: 8-15 bpm'),
    ('summary.awake_minutes', 'Awake Minutes', 'Expected: ~960 (16hrs)'),
    ('summary.asleep_grpd_rstls', 'Asleep Minutes', 'Expected: ~420-480 (7-8hrs)'),
    ('summary.sleep_start_time', 'Sleep Start (minute of day)', 'Expected: ~1300-1400 (9-11pm)'),
    ('summary.pct_awake_hr_low', '% Awake Low HR', 'Expected: ~70-80%'),
    ('summary.pct_awake_hr_med', '% Awake Med HR', 'Expected: ~15-20%'),
    ('summary.pct_awake_hr_high', '% Awake High HR', 'Expected: ~3-8%'),
    ('summary.age', 'Age', 'Expected: 18-90'),
    ('bmi', 'BMI', 'Expected: 18-40 avg'),
    ('is_male', 'Is Male', 'Expected: 0 or 1'),
]
for idx, (col, title, note) in enumerate(plots):
    ax = axes[idx // 4][idx % 4]
    data = daily_activity_stats_ehr[col].dropna()
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{title}\n({note})', fontsize=10)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'median={data.median():.1f}')
    ax.legend(fontsize=8)
# Awake + Asleep check
ax = axes[3][2]
total = daily_activity_stats_ehr['summary.awake_minutes'] + daily_activity_stats_ehr['summary.asleep_grpd_rstls']
ax.hist(total.dropna(), bins=50, color='green', edgecolor='white', alpha=0.8)
ax.set_title('Awake + Asleep Total\n(Should be exactly 1440)', fontsize=10)
ax.axvline(1440, color='red', linestyle='--', label='1440 min')
ax.legend(fontsize=8)
# Hide unused subplot
axes[3][3].axis('off')
plt.tight_layout()
plt.savefig('latent_validation.png', dpi=150, bbox_inches='tight')
plt.show()
# Summary table
print("\n=== Quick Validation Summary ===")
print(f"Total rows: {len(daily_activity_stats_ehr)}")
print(f"Unique people: {daily_activity_stats_ehr['person_id'].nunique()}")
for col, title, note in plots:
    data = daily_activity_stats_ehr[col]
    print(f"\n{title}:")
    print(f"  NaN rate: {data.isna().mean()*100:.1f}%  |  mean: {data.mean():.2f}  |  median: {data.median():.2f}")
# HR bucket sanity
hr_sum = (daily_activity_stats_ehr['summary.pct_awake_hr_low'] + 
          daily_activity_stats_ehr['summary.pct_awake_hr_med'] + 
          daily_activity_stats_ehr['summary.pct_awake_hr_high'])
print(f"\n=== HR Bucket Sum Check ===")
print(f"  Mean sum: {hr_sum.mean():.6f} (should be 1.0)")
print(f"  All equal 1.0: {(hr_sum.dropna() == 1.0).all()}")